In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('..')
from src.models import LinearRegression
from src.metrics import ECM	

In [ ]:
dev_original = pd.read_csv("../data/casas_dev.csv")

1.1

Fragmento random del dataset

In [ ]:
dev_original.sample(10)

Variables posibles dentro de unidades y tipo

In [ ]:
print(dev_original['tipo'].unique())
print(dev_original['unidades'].unique())


Lo uso porque de esta manera me aseguro que no haya otros valores posibles para luego categorizar / hacer one hot encodding en el caso del tipo.

Analisis a simple vista:
- Columna unidades: deberia estar en una misma medida, elijo m2. Esto es para que sean comparables las medidas.
- Columna pisos: demasiados NaN
- Columna pileta: convertir en 1(True) y 0(False). Elijo tener todo numericamente
- Columna tipo: pasar de variable categorica a numero
- Normalizar las variables (las escalas son demasiado distintas)

Busco NaNs en cada columna:

In [ ]:
columnas = []
porcentajes = []
for col in dev_original.columns:
	columna = dev_original[col]
	total_NaNs_en_columna = columna.isna().sum() #sumo cantidad de NaNs 
	porcentaje_NaNs_en_columna = (total_NaNs_en_columna/dev_original.shape[0]) * 100

	columnas.append(col)
	porcentajes.append(porcentaje_NaNs_en_columna)

plt.figure()
plt.bar(columnas, porcentajes)
plt.xticks(rotation=45)
plt.ylabel("Porcentaje de NaNs")
plt.title("Porcentaje de valores faltantes por columna")
plt.tight_layout()
plt.show()


- Para los NaNs de edad usare la media de esa feature para reemplazarlos.
- Para los de precio hare lo mismo.
- En ambos casos calculare la media contemplando su ubicacion, para que la estimacion de esos valores sea lo mas adecuada posible. (esto lo hare mas adelante porque primero debo separar en train y validation para no generar data l)

- Decido eliminar la columna de pisos porque hay mas de un 50% de valores faltantes, lo que puede producir sesgos importantes. 

- Las demas no tienen  porcentajes tan significativos 
	--> utilizo media para estimar valores numericos y reemplazarlos.


Antes de eliminar veo la correlacion de precios con pisos (en los valores que no son NaN), para tomar desicion final

In [ ]:
correlacion_pisos_precios = dev_original['precio'].corr(dev_original['pisos'])
print(correlacion_pisos_precios)

Efectivamente es conveniente eliminar la columna de pisos. Ademas de los valores faltantes, la correlacion entre precios y pisos (con los valores que si existe) es muy debil.

In [ ]:
dev = dev_original.drop('pisos', axis=1) #elimina columna pisos

Todos los cambios desde ahora en dev los hare sobre una copia para luego usar el dev_original

Identifico como estan distribuidas las casas en lat y long

In [ ]:
plt.figure()

lat = dev['lat']
lon = dev['lon']
plt.scatter(lat,lon,s=50, alpha=0.3)
plt.xlabel("lat")
plt.ylabel("lon")
plt.title("Distribucion latitud y longitud")

plt.show()

Se puede ver que la ubicacion de las casas (en lat y lon) estan concentradas unicamente en dos lugares:
Mirando el grafico se puede ver que estas son: 1 --> (-34, -58): buscandolo en el mapa equivale a la provincia de Buenos Aires (Argentina) y 2 --> (40,-74): buscandolo en el mapa equivale a New York (USA)

Agrego Feature nueva Ubicacion que reemplaza lat y long. 
Los valores seran 1 si la ubicacion es New York y 0 si es Buenos Aires (para no usar variables categoricas)

In [ ]:
dev['Ubicacion'] = (dev['lat'] > 0).astype(int) #crea una columna nueva fijandose por la latitud si es New York o Argentina (solo hace falta ver latitud)

Elimino lat y long porque ubicacion las reemplazo

In [ ]:
dev = dev.drop('lat',axis=1)
dev = dev.drop('lon',axis=1)

Cambio columna pileta a 1/0 en vez de True/False

In [ ]:
dev['pileta'] = (dev['pileta']).astype(int)

- Convierto sqft en m2.
- Cuando la variable en Area/metros cubiertos este escrita en sqft, la paso a metros

In [ ]:
dev.loc[dev['Ubicacion'] == 1, 'Área'] = dev.loc[dev['Ubicacion'] == 1, 'Área'] / 10.764 
dev.loc[dev['Ubicacion'] == 1, 'metros_cubiertos'] = dev.loc[dev['Ubicacion'] == 1, 'metros_cubiertos'] / 10.764 

Ahora elimino la columna unidades porque ya esta unificado en m2

In [ ]:
dev = dev.drop('unidades',axis=1)

Tengo que cambiar tipo a variables numericas porque es categorica:
Uso One-Hot Encodding

In [ ]:
dev['es_casa'] = (dev['tipo'] == 'casa').astype(int)
dev['es_ph'] = (dev['tipo'] == 'ph').astype(int)

Si no es casa ni ph, es dept

Elimino tipo porque no me sirve

In [ ]:
dev = dev.drop('tipo', axis=1)

Analizo rango de precios en Argentina

In [ ]:
print(dev.loc[dev['Ubicacion'] == 0, 'precio'].describe())

Analizo rango de precios en USA

In [ ]:
print(dev.loc[dev['Ubicacion'] == 1, 'precio'].describe())

Tomo en cuenta que hay algo raro en los precios de Argentina

CHECK DE CAMBIOS EN EL DATASET

In [ ]:
dev.sample(5)

1.2

Analisis con Boxplot

In [ ]:
plt.figure()
plt.boxplot(dev.loc[dev['Ubicacion'] == 0, 'precio'].dropna())
plt.title('Boxplot Precio - Buenos Aires')
plt.show()

plt.figure()
plt.boxplot(dev.loc[dev['Ubicacion'] == 1, 'precio'].dropna())
plt.title('Boxplot Precio - Nueva York')
plt.show()
for col in dev.select_dtypes(include=['number']).columns:
	if col != 'precio' and col != 'Ubicacion' and col != 'pileta' and col != 'es_casa' and col != 'es_ph':
		plt.figure()
		dev.boxplot(column=col)
		plt.title(f'Boxplot de {col}')
		plt.show()

In [ ]:
dev['pileta'].value_counts(normalize=True)*100

Un 82% de casas no tiene pileta pero hay un 18% que si. La feature puede aportar algo al modelo.

Analisis Boxplots (tomando en cuenta lo importante): 

- Precios: 
Buenos Aires: Concentrados entre 0 y 20000 con outliers arriba de 80000. Sesgo hacia valores altos.
Nueva York: Concentrados entre 100000 y 250000 con outliers arriba de 400000 y cercanos a 0.

- Area:
Concentrada entre 50 y 200 con outliers arriba de 250.

- Metros Cubiertos:
Concentrado entre 75 y 125 pero con outliers arriba de 200

- Ambientes:
Rango entre 1 y 10. Distribucion pareja.

- Edad:
Valores concentrados entre 0 y 40 años, otliers arriba de 80 años, generando sesgo hacia casas nuevas.



- Los outliers pueden distorsionar el modelo. 
- Los precios entre Buenos Aires y Nueva York estan en escalas muy distintas. Ademas, en cada uno hay valores muy cercanos a 0. Los que sean 0 seran detectados como error y los que no habria que analizar para detectar si pueden ser alquileres.
- La normalizacion lleva todas las variables a una misma escala, permitiendo que el algoritmo de gradiente descendiente converja de forma mas rapida y estable.


In [ ]:
for col in dev.select_dtypes(include=['number']).columns:
    if col != 'precio' and col != 'es_casa' and col != 'es_ph':
        plt.figure()
        sns.scatterplot(data=dev, x=col, y='precio', hue='Ubicacion')
        plt.title(f'precio vs {col}')
        plt.show()

Relaciones importantes:
- Precio vs Area/Metro cubiertos:
Cuanto mas terreno/espacio --> mas alto el precio

- Precio vs Pileta:
En Buenos Aires aumenta el precio al tener pileta (por clima)

-Precio vs Ubicacion:
Claramente hay diferencias de escala por la moneda.


- Precio vs ambientes: 
Telación poco clara

- Precio vs edad: 
No depende tanto


1.3

Separo dataset (80% train - 20% validation)

In [ ]:
from src.data_splitting import train_val_split	

In [ ]:
train_set,validation_set = train_val_split(dev,42)

Normalizacion de variables

Normalizo en memoria (no cambio csv) --> Quiero mantener dataset original

In [ ]:
train_original = pd.read_csv("../data/casas_train.csv")
validation_original = pd.read_csv("../data/casas_validation.csv")

In [ ]:
train = train_original
validation = validation_original

Reemplazo de NaNs

In [ ]:
media_precios_NYC = (train.loc[train['Ubicacion'] == 1, 'precio']).mean()
media_precios_BUE = (train.loc[train['Ubicacion'] == 0, 'precio']).mean()
media_edad_NYC = (train.loc[train['Ubicacion'] == 1, 'edad']).mean()
media_edad_BUE = (train.loc[train['Ubicacion'] == 0, 'edad']).mean()

for parte in [train,validation]:
	parte.loc[(parte['precio'].isna()) & (parte['Ubicacion'] == 1), 'precio'] = media_precios_NYC
	parte.loc[(parte['precio'].isna()) & (parte['Ubicacion'] == 0), 'precio'] = media_precios_BUE
	parte.loc[(parte['edad'].isna()) & (parte['Ubicacion'] == 1), 'edad'] = media_edad_NYC
	parte.loc[(parte['edad'].isna()) & (parte['Ubicacion'] == 0), 'edad'] = media_edad_BUE

In [ ]:
from src.preprocessing import normalizar
train_normalizado, validation_normalizado = normalizar(['precio','Área','metros_cubiertos', 'ambientes','edad'],train,validation)

Visualizacion de variables normalizadas:

In [ ]:
train_normalizado.sample(5)

In [ ]:
validation_normalizado.sample(5)

In [ ]:
pd.set_option('display.float_format', '{:.4f}'.format)
print("Antes de normalizar:")
print(pd.read_csv("../data/casas_train.csv").describe())
print("\nDespués de normalizar:")
print(train_normalizado.describe())

Todo se normalizo correctamente

EN QUE MOMENTO TENGO QUE SACAR LA NORMALIZACION?




EJERCICIO 2

In [ ]:
nombres_features = train_normalizado.drop('precio',axis=1).columns

X_train_todas_vars = train_normalizado.drop('precio',axis=1).values

X_train_una_var = train_normalizado['Área'].values

y_train_norm = train_normalizado['precio'].values
y_validation_norm = validation_normalizado['precio'].values

Entrenamiento con Pseudo Inversa

Todas las variables

In [ ]:
modelo_pi_todas = LinearRegression(X_train_todas_vars, y_train_norm,nombres_features)
modelo_pi_todas.entrenar_pseudo_inv()
modelo_pi_todas.coefs_con_features()
y_pred_pi_todas = modelo_pi_todas.X @ modelo_pi_todas.w
print("ECM:", ECM(y_train_norm, y_pred_pi_todas))

Una variable

In [ ]:
modelo_pi_1 = LinearRegression(X_train_una_var, y_train_norm,nombres_features)
modelo_pi_1.entrenar_pseudo_inv()
modelo_pi_1.coefs_con_features()
y_pred_pi_1 = modelo_pi_1.X @ modelo_pi_1.w
print("ECM:", ECM(y_train_norm, y_pred_pi_1))

Entrenamiento con Gradiente Descendiente

In [ ]:
alfa = 0.1
iteraciones = 1000

In [ ]:
modelo_gd_todas = LinearRegression(X_train_todas_vars, y_train_norm,nombres_features)
modelo_gd_todas.entrenar_gradiente_descendiente(alfa,iteraciones)
modelo_gd_todas.coefs_con_features()
y_pred_gd_todas = modelo_gd_todas.X @ modelo_gd_todas.w
print("ECM:", ECM(y_train_norm, y_pred_gd_todas))

Una variable

In [ ]:
modelo_gd_1 = LinearRegression(X_train_una_var, y_train_norm,nombres_features)
modelo_gd_1.entrenar_gradiente_descendiente(alfa,iteraciones)
modelo_gd_1.coefs_con_features()
y_pred_gd_1 = modelo_gd_1.X @ modelo_gd_1.w
print("ECM:", ECM(y_train_norm, y_pred_gd_1))

Implementacion correcta, modelos dan resultados similares

EJERCICIO 3

In [ ]:
Modelo_1 = modelo_pi_1 #uso el mismo que use en el ej anterior
Modelo_1.coefs_con_features()
print("ECM:", ECM(y_train_norm, y_pred_pi_1))

In [ ]:
X_train_2 = train_normalizado[['Área','pileta']].values
Modelo_2 = LinearRegression(X_train_2, y_train_norm,train_normalizado[['Área', 'pileta']].columns)
Modelo_2.entrenar_pseudo_inv()
Modelo_2.coefs_con_features()
y_pred_M2 = Modelo_2.X @ Modelo_2.w
print("ECM:", ECM(y_train_norm, y_pred_M2))

El valor del peso indica que la pileta casi que no afecta el precio de la propiedad (-0.0974). Esto en realidad es porque al no tener en cuenta la ubicacion y la distorsion en los precios con USA, que lo sesgan. En realidad el peso de la pileta deberia ser nulo o levemente positivo.

En el 3.3 sacaria es_ph (ya tengo es_casa) y pileta porque no aportan mucho

In [ ]:
X_train_3 = train_normalizado[['Área','metros_cubiertos','edad','Ubicacion','es_casa','ambientes']].values
Modelo_3 = LinearRegression(X_train_3, y_train_norm,train[['Área','metros_cubiertos','edad','Ubicacion','es_casa','ambientes']].columns)
Modelo_3.entrenar_pseudo_inv()
y_pred_M3 = Modelo_3.X @ Modelo_3.w
Modelo_3.coefs_con_features()
print("ECM:", ECM(y_train_norm, y_pred_M3))

Hay pesos que son extraños, por ejemplo peso de metros cubiertos negativo. Esto puede ser porque al ser una feature tan correlacionada con Area, produce inestabilidad (no es clara la contribucion de cada uno).

EJERCICIO 4

Las features a agregar son:
- metros_cubiertos / area --> Me dicen proporcion construida del terreno
- pileta x Ubicacion --> Vimos que el que haya pileta, influye mas si es en Bs.As. que en NYC. --> 1 si tiene pileta y es de NYC (1 x 1) y 0 caso contrario.
- metros_cubiertos / ambientes --> saber el tamaño promedio por ambiente --> si los ambientes son mas amplios la casa podria valer mas


In [ ]:
from src.feature_engeneering import crear_features

In [ ]:
crear_features (train)
crear_features(validation)
cols_M4_a_norm = ['precio','Área','metros_cubiertos','edad','ambientes',
           'metros_cubiertos / Área','metros_cubiertos / ambientes'] #no normalizo las columnas binarias 

train_normalizado,validation_normalizado = normalizar(cols_M4_a_norm,train,validation)

In [ ]:
X_train_4 = train_normalizado.drop(['precio','es_ph','pileta'],axis=1).values
y_train_norm = train_normalizado['precio']
Modelo_4 = LinearRegression(X_train_4,y_train_norm,train_normalizado.drop(['precio','es_ph','pileta'],axis=1).columns)
Modelo_4.entrenar_pseudo_inv()
y_pred_M4 = Modelo_4.X @ Modelo_4.w
print("ECM:",ECM(y_train_norm,y_pred_M4))


FEATURES POLINOMICAS

In [ ]:
from src.feature_engeneering import crear_features_polinomicas

In [ ]:
grado_max = 7
cols_base = train.drop(['es_ph','pileta','Ubicacion','es_casa','pileta x Ubicacion'], axis=1).columns #no voy a elevar las binarias porque no cambia nada
crear_features_polinomicas(train, cols_base, grado_max)
crear_features_polinomicas(validation, cols_base, grado_max)

cols_a_norm = train.drop(['es_ph','pileta','Ubicacion','es_casa','pileta x Ubicacion'], axis=1).columns
train_normalizado_M5, validation_normalizado_M5 = normalizar(cols_a_norm, train, validation)

In [ ]:
X_train_5 = train_normalizado_M5.drop(['precio','es_ph','pileta'],axis=1).values #estoy usando las nuevas mas la de el modelo 3
y_train_norm = train_normalizado_M5['precio']
Modelo_5 = LinearRegression(X_train_5,y_train_norm,train_normalizado_M5.drop(['es_ph','pileta'],axis=1).columns)
Modelo_5.entrenar_pseudo_inv()
y_pred_M5 = Modelo_5.X @ Modelo_5.w
print("ECM:",ECM(y_train_norm,y_pred_M5))

In [ ]:
X_validation_5 = validation_normalizado_M5.drop(['precio','es_ph','pileta'],axis=1).values
X_val = np.column_stack((np.ones(len(X_validation_5)), X_validation_5))
y_pred_M5_val = X_val @ Modelo_5.w
y_validation_norm = validation_normalizado_M5['precio'] 
print("ECM:",ECM(y_validation_norm,y_pred_M5_val))

Con estos experimentos esperaba que a medida que se aumente la cantidad de features, mejor predeciri (que fue lo que sucedio). Igualmente como es de esperarse, si se continua agregando features (principalmente polinomicas), el modelo comenzaria a overfittear debido a que estaria apoyandose demasiado en el train set, teniendo mala performance en el validation.